In [1]:
import os
import subprocess
from tqdm import tqdm

def extract_meld_audio_ffmpeg(base_path, output_root):
    """
    Extracts mono audio from MELD mp4 files using FFmpeg.
    Standardizes output to 16kHz wav for Wav2Vec2 compatibility.
    """
    sub_folders = ['train', 'dev', 'test']
    
    for folder in sub_folders:
        input_dir = os.path.join(base_path, folder)
        output_dir = os.path.join(output_root, folder)
        
        # Create output sub-directories if they don't exist
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"📁 Created directory: {output_dir}")

        if not os.path.exists(input_dir):
            print(f"⚠️ Skipping {folder}: Source path not found.")
            continue

        print(f"🚀 Extracting audio from {folder} via FFmpeg...")
        video_files = [f for f in os.listdir(input_dir) if f.endswith('.mp4')]
        
        for video_file in tqdm(video_files):
            video_path = os.path.normpath(os.path.join(input_dir, video_file))
            # Convert filename extension from .mp4 to .wav
            audio_file_name = video_file.replace('.mp4', '.wav')
            audio_output_path = os.path.normpath(os.path.join(output_dir, audio_file_name))
            
            # Skip if file already exists (useful for resuming interrupted tasks)
            if os.path.exists(audio_output_path):
                continue

            # FFmpeg Command breakdown:
            # -y: Overwrite output files without asking
            # -i: Input file path
            # -vn: Disable video recording (extract audio only)
            # -ac 1: Set audio channels to 1 (Mono - critical for 5.1 audio sources)
            # -ar 16000: Set audio sampling rate to 16kHz (Standard for Wav2Vec2)
            # -loglevel error: Suppress non-critical logs
            command = [
                'ffmpeg',
                '-y',
                '-i', video_path,
                '-vn',
                '-ac', '1',
                '-ar', '16000',
                audio_output_path,
                '-loglevel', 'error'
            ]

            try:
                # Execute the shell command
                subprocess.run(command, check=True)
            except subprocess.CalledProcessError as e:
                print(f"❌ File corrupted or FFmpeg error at {video_file}: {e}")
            except FileNotFoundError:
                print("🚨 FFmpeg not found! Please ensure it's installed and added to PATH.")
                print("Hint: Run 'conda install ffmpeg' or 'sudo apt install ffmpeg'.")
                return

# Execution
if __name__ == "__main__":
    extract_meld_audio_ffmpeg(
        base_path="./MELD_Vid", 
        output_root="./Raw_Data/MELD"
    )

📁 Created directory: ./Raw_Data/MELD/train
🚀 Extracting audio from train via FFmpeg...


 56%|█████▋    | 5643/9989 [03:45<02:56, 24.62it/s][mov,mp4,m4a,3gp,3g2,mj2 @ 0x5cdc61e6af40] moov atom not found
[in#0 @ 0x5cdc61e6ae40] Error opening input: Invalid data found when processing input
Error opening input file MELD_Vid/train/dia125_utt3.mp4.
Error opening input files: Invalid data found when processing input
 57%|█████▋    | 5649/9989 [03:46<03:03, 23.63it/s]

❌ File corrupted or FFmpeg error at dia125_utt3.mp4: Command '['ffmpeg', '-y', '-i', 'MELD_Vid/train/dia125_utt3.mp4', '-vn', '-ac', '1', '-ar', '16000', 'Raw_Data/MELD/train/dia125_utt3.wav', '-loglevel', 'error']' returned non-zero exit status 183.


100%|██████████| 9989/9989 [06:40<00:00, 24.94it/s]


📁 Created directory: ./Raw_Data/MELD/dev
🚀 Extracting audio from dev via FFmpeg...


100%|██████████| 1112/1112 [00:43<00:00, 25.37it/s]


📁 Created directory: ./Raw_Data/MELD/test
🚀 Extracting audio from test via FFmpeg...


100%|██████████| 2747/2747 [01:50<00:00, 24.92it/s]


In [3]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

class VoxSentinelDataProtocol:
    def __init__(self, root_path="./Raw_Data"):
        self.root_path = root_path
        self.clean_data = []  # 存放 TESS, RAVDESS, CREMA
        self.meld_data = []   # 专门存放 MELD，待后续切分
        
        self.emotion_map = {
            'joy': 'happy', 'sadness': 'sad', 'anger': 'angry', 
            'fear': 'fear', 'disgust': 'disgust', 'surprise': 'surprise', 'neutral': 'neutral',
            'HAP': 'happy', 'SAD': 'sad', 'ANG': 'angry', 
            'FEA': 'fear', 'DIS': 'disgust', 'NEU': 'neutral',
            'happy': 'happy', 'sad': 'sad', 'angry': 'angry', 'ps': 'surprise',
            'pleasant_surprised': 'surprise'
        }

    def _stratified_split(self, data_list, train_size=0.8, dev_size=0.1):
        if not data_list: return []
        df = pd.DataFrame(data_list)
        train_df, temp_df = train_test_split(df, test_size=(1 - train_size), stratify=df['emotion'], random_state=42)
        relative_dev_size = dev_size / (1 - train_size)
        dev_df, test_df = train_test_split(temp_df, test_size=(1 - relative_dev_size), stratify=temp_df['emotion'], random_state=42)
        train_df['split'], dev_df['split'], test_df['split'] = 'train', 'dev', 'test'
        return pd.concat([train_df, dev_df, test_df]).to_dict('records')

    # --- 数据处理函数 (CREMA, RAVDESS, TESS 存入 clean_data) ---
    def process_crema(self):
        print("🔍 Processing CREMA-D...")
        folder = os.path.join(self.root_path, "Crema")
        temp = []
        if os.path.exists(folder):
            for file in os.listdir(folder):
                if file.endswith(".wav"):
                    parts = file.split('_')
                    if len(parts) > 2 and parts[2] in self.emotion_map:
                        temp.append({'path': os.path.join(folder, file), 'emotion': self.emotion_map[parts[2]], 'dataset': 'crema'})
        self.clean_data.extend(self._stratified_split(temp))

    def process_ravdess(self):
        print("🔍 Processing RAVDESS...")
        rav_root = os.path.join(self.root_path, "RAVDESS")
        rav_map = {'01':'neutral', '03':'happy', '04':'sad', '05':'angry', '06':'fear', '07':'disgust', '08':'surprise'}
        temp = []
        if os.path.exists(rav_root):
            for root, _, files in os.walk(rav_root):
                for file in files:
                    if file.endswith(".wav"):
                        parts = file.split('-')
                        if len(parts) > 2 and parts[2] in rav_map:
                            temp.append({'path': os.path.join(root, file), 'emotion': rav_map[parts[2]], 'dataset': 'ravdess'})
        self.clean_data.extend(self._stratified_split(temp))

    def process_tess(self):
        print("🔍 Processing TESS...")
        tess_root = os.path.join(self.root_path, "Tess")
        temp = []
        if os.path.exists(tess_root):
            for folder in os.listdir(tess_root):
                folder_path = os.path.join(tess_root, folder)
                if os.path.isdir(folder_path):
                    raw_emo = folder.split('_')[-1].lower()
                    for file in os.listdir(folder_path):
                        if file.endswith(".wav") and raw_emo in self.emotion_map:
                            temp.append({'path': os.path.join(folder_path, file), 'emotion': self.emotion_map[raw_emo], 'dataset': 'tess'})
        self.clean_data.extend(self._stratified_split(temp))

    def process_meld(self):
        print("🔍 Processing MELD (Collecting for re-splitting)...")
        meld_root = os.path.join(self.root_path, "MELD")
        splits = {'train': 'train_sent_emo.csv', 'dev': 'dev_sent_emo.csv', 'test': 'test_sent_emo.csv'}
        for split_name, csv_filename in splits.items():
            csv_path = os.path.join(meld_root, csv_filename)
            audio_subdir = os.path.join(meld_root, split_name)
            if not os.path.exists(csv_path): continue
            df = pd.read_csv(csv_path)
            for _, row in df.iterrows():
                file_name = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.wav"
                audio_path = os.path.normpath(os.path.join(audio_subdir, file_name))
                raw_emo = str(row['Emotion']).lower().strip()
                if os.path.exists(audio_path) and raw_emo in self.emotion_map:
                    self.meld_data.append({
                        'path': audio_path, 'emotion': self.emotion_map[raw_emo],
                        'dataset': 'meld', 'original_split': split_name
                    })

    def export(self, output_dir="./Protocol"):
        if not os.path.exists(output_dir): 
            os.makedirs(output_dir)
        
        meld_df = pd.DataFrame(self.meld_data)
        
        # 1. 拆分逻辑 (20% Master, 40% FineTune, 40% External)
        main_meld, remain_meld = train_test_split(
            meld_df, test_size=0.8, stratify=meld_df['emotion'], random_state=42
        )
        finetune_meld, external_meld = train_test_split(
            remain_meld, test_size=0.5, stratify=remain_meld['emotion'], random_state=42
        )
        
        # --- A. Master Protocol ---
        main_meld_renamed = main_meld.rename(columns={'original_split': 'split'})
        # 🚀 修复核心：这里加上 .reset_index(drop=True) 确保索引唯一
        master_df = pd.concat([pd.DataFrame(self.clean_data), main_meld_renamed]).reset_index(drop=True)
        master_df.to_csv(os.path.join(output_dir, "master_metadata.csv"), index=False)
        
        # --- B. FineTune Protocol ---
        ft_list = self._stratified_split(finetune_meld.to_dict('records'))
        ft_df = pd.DataFrame(ft_list)
        ft_df.to_csv(os.path.join(output_dir, "finetune_metadata.csv"), index=False)
        
        # --- C. External Test Protocol ---
        # 🚀 这里也建议加上 reset_index，防止后续评估时报同样的错
        ext_df = external_meld.rename(columns={'original_split': 'split'}).reset_index(drop=True)
        ext_df.to_csv(os.path.join(output_dir, "external_test_metadata.csv"), index=False)

        # =============================================================
        # 📊 详细报告打印 (现在索引唯一了，crosstab 会运行顺畅)
        # =============================================================
        print("\n" + "="*60)
        print("📁 PROTOCOL GENERATION SUMMARY")
        print("="*60)
        print(f"Total Master Samples   : {len(master_df)}")
        print(f"Total FineTune Samples : {len(ft_df)}")
        print(f"Total External Samples : {len(ext_df)}")
        
        print("\n" + "="*60)
        print("🎯 MASTER PROTOCOL: DEEP DIVE")
        print("="*60)
        
        # 1. 数据集构成
        print("\n--- [1] Composition by Dataset ---")
        ds_counts = master_df['dataset'].value_counts()
        for ds, count in ds_counts.items():
            percentage = (count / len(master_df)) * 100
            print(f"📍 {ds:<10}: {count:>5} samples ({percentage:>5.1f}%)")

        # 2. 情感分布
        print("\n--- [2] Emotion Distribution (Master) ---")
        # 索引唯一后，这一行将完美运行
        emo_dist = pd.crosstab(master_df['emotion'], master_df['split'])
        print(emo_dist)

        # 3. 比例统计
        print("\n--- [3] Split Ratios (Master) ---")
        split_counts = master_df['split'].value_counts(normalize=True) * 100
        for split, per in split_counts.items():
            print(f"🔸 {split:<6}: {per:>5.1f}%")

        print("\n" + "="*60)
        print("🛠️ FINETUNE & EXTERNAL PREVIEW")
        print("="*60)
        print(f"FineTune Top Emotions:\n{ft_df['emotion'].value_counts(normalize=True).head(3)}")
        print(f"\nExternal Top Emotions:\n{ext_df['emotion'].value_counts().head(3)}")
        print("="*60)

if __name__ == "__main__":
    protocol = VoxSentinelDataProtocol()
    protocol.process_crema()
    protocol.process_ravdess()
    protocol.process_tess()
    protocol.process_meld()
    protocol.export()

🔍 Processing CREMA-D...
🔍 Processing RAVDESS...
🔍 Processing TESS...
🔍 Processing MELD (Collecting for re-splitting)...

📁 PROTOCOL GENERATION SUMMARY
Total Master Samples   : 14031
Total FineTune Samples : 5482
Total External Samples : 5483

🎯 MASTER PROTOCOL: DEEP DIVE

--- [1] Composition by Dataset ---
📍 crema     :  7442 samples ( 53.0%)
📍 meld      :  2741 samples ( 19.5%)
📍 tess      :  2600 samples ( 18.5%)
📍 ravdess   :  1248 samples (  8.9%)

--- [2] Emotion Distribution (Master) ---
split     dev  test  train
emotion                   
angry     214   248   1722
disgust   191   198   1546
fear      193   196   1546
happy     217   250   1858
neutral   239   423   2208
sad       208   227   1628
surprise   64    89    566

--- [3] Split Ratios (Master) ---
🔸 train :  78.9%
🔸 test  :  11.6%
🔸 dev   :   9.5%

🛠️ FINETUNE & EXTERNAL PREVIEW
FineTune Top Emotions:
emotion
neutral     0.469354
happy       0.168369
surprise    0.119300
Name: proportion, dtype: float64

External Top